In [2]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities import extractor
import uproot
import awkward as ak    

x_MH25=extractor("Dati/Tprime_tAq_1800_MH25_LH_2017.root", "Events")


file=uproot.open("Dati/Tprime_tAq_1800_MH25_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]

#Filtriamo i dati

mask = ak.flatten(Fatjet_isMatchedWithA) == 1
x_filtered = x_MH25[mask]



/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /home/riccardo/anaconda3/envs/rootnev/include/site/python3.14); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "
/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


In [3]:
from scipy.special import voigt_profile
from scipy.optimize import curve_fit
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()

def voigt(x, norm, mu, sigma, gamma):
    return voigt_profile(x-mu, sigma, gamma) * norm

bin_counts, bin_edges = np.histogram(x_plot, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_plot) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_plot) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt)

m_voigt=Minuit(ls_voigt,  norm=1, mu=50, sigma=5, gamma=1)
m_voigt.limits["mu"]= (10, 40)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.migrad()








┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 3166 (χ²/ndof = 68.8)      │              Nfcn = 177              │
│ EDM = 5.63e-05 (Goal: 0.0002)    │            time = 0.4 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm  │   0.951   │   0.004   │            │            │         │         │       │
│ 1 │ mu    │  25.550   │   0.017   │            │            │   10    │   40    │       │
│ 2 │ sigma │   2.844   │   0.023   │            │            │   0.1   │   20    │       │
│ 3 │ gamma │   0.284   │   0.011   │            │            │  0.01   │   10    │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬─────────────────────────────────────────┐
│       │      norm        mu     sigma     gamma │
├───────┼─────────────────────────────────────────┤
│  norm │   1.7e-05  0.003e-3 -0.008e-3  0.003e-3 │
│    mu │  0.003e-3  0.000307  -0.25e-3   0.06e-3 │
│ sigma │ -0.008e-3  -0.25e-3  0.000521  -0.17e-3 │
│ gamma │  0.003e-3   0.06e-3  -0.17e-3  0.000115 │
└───────┴─────────────────────────────────────────┘

In [8]:
import pickle

with open("fit_results.pkl", "wb") as f:
    pickle.dump(m_voigt.values, f)

print(m_voigt.values)

print(m_voigt.errors)
print(m_voigt.parameters)

<ValueView norm=0.9508152528989859 mu=25.54969272049765 sigma=2.844057736211518 gamma=0.284418290713378>
<ErrorView norm=0.004119833545233603 mu=0.017533304649871795 sigma=0.0228152570266984 gamma=0.010743581189916168>
('norm', 'mu', 'sigma', 'gamma')


In [23]:
fit_values={'MH25': fit_MH25_values,}
fit_errors={'MH25_errors': fit_MH25_errors}

fit_MH25_values={}
fit_MH25_errors={}

for param in m_voigt.parameters:
    fit_MH25_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deve estarre gli errori 
    fit_MH25_errors[error] = m_voigt.errors[error]

print(fit_MH25_values)
print(fit_MH25_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH25"]=fit_MH25_values
results["MH25_errors"]=fit_MH25_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH25"]=fit_MH25_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH25_errors"]=fit_MH25_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  





{'norm': 0.9508152528989859, 'mu': 25.54969272049765, 'sigma': 2.844057736211518, 'gamma': 0.284418290713378}
{'norm': 0.004119833545233603, 'mu': 0.017533304649871795, 'sigma': 0.0228152570266984, 'gamma': 0.010743581189916168}
